![Two data scientists working on a dashboard.](hr-image-small.png)

A common problem when creating models to generate business value from data is that the datasets can be so large that it can take days for the model to generate predictions. Ensuring that your dataset is stored as efficiently as possible is crucial for allowing these models to run on a more reasonable timescale without having to reduce the size of the dataset.

You've been hired by a major online data science training provider called *Training Data Ltd.* to clean up one of their largest customer datasets. This dataset will eventually be used to predict whether their students are looking for a new job or not, information that they will then use to direct them to prospective recruiters.

You've been given access to `customer_train.csv`, which is a subset of their entire customer dataset, so you can create a proof-of-concept of a much more efficient storage solution. The dataset contains anonymized student information, and whether they were looking for a new job or not during training:

| Column                   | Description                                                                      |
|------------------------- |--------------------------------------------------------------------------------- |
| `student_id`             | A unique ID for each student.                                                    |
| `city`                   | A code for the city the student lives in.                                        |
| `city_development_index` | A scaled development index for the city.                                         |
| `gender`                 | The student's gender.                                                            |
| `relevant_experience`    | An indicator of the student's work relevant experience.                          |
| `enrolled_university`    | The type of university course enrolled in (if any).                              |
| `education_level`        | The student's education level.                                                   |
| `major_discipline`       | The educational discipline of the student.                                       |
| `experience`             | The student's total work experience (in years).                                  |
| `company_size`           | The number of employees at the student's current employer.                       |
| `company_type`           | The type of company employing the student.                                       |
| `last_new_job`           | The number of years between the student's current and previous jobs.             |
| `training_hours`         | The number of hours of training completed.                                       |
| `job_change`             | An indicator of whether the student is looking for a new job (`1`) or not (`0`). |

In [44]:
# Import necessary libraries
import pandas as pd

# Load the dataset
ds_jobs = pd.read_csv("customer_train.csv")

# View the dataset
ds_jobs.head()

,student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
0,8949,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevant experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevant experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevant experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevant experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


In [45]:
# Create a copy of ds_jobs for transforming
ds_jobs_transformed = ds_jobs.copy()

# Start coding here. Use as many cells as you like!

# Exploratory data analysis

In [46]:
ds_jobs_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              19158 non-null  int64  
 1   city                    19158 non-null  object 
 2   city_development_index  19158 non-null  float64
 3   gender                  14650 non-null  object 
 4   relevant_experience     19158 non-null  object 
 5   enrolled_university     18772 non-null  object 
 6   education_level         18698 non-null  object 
 7   major_discipline        16345 non-null  object 
 8   experience              19093 non-null  object 
 9   company_size            13220 non-null  object 
 10  company_type            13018 non-null  object 
 11  last_new_job            18735 non-null  object 
 12  training_hours          19158 non-null  int64  
 13  job_change              19158 non-null  float64
dtypes: float64(2), int64(2), object(10)
me

In [47]:
ds_jobs_transformed.dtypes

student_id                  int64
city                       object
city_development_index    float64
gender                     object
relevant_experience        object
enrolled_university        object
education_level            object
major_discipline           object
experience                 object
company_size               object
company_type               object
last_new_job               object
training_hours              int64
job_change                float64
dtype: object

# Converting integers, floats, and unordered categories

In [48]:
# Convert integer columns to int32
int_columns = ds_jobs_transformed.select_dtypes(include=['int64']).columns
ds_jobs_transformed[int_columns] = ds_jobs_transformed[int_columns].astype('int32')

# Convert float columns to float16
float_columns = ds_jobs_transformed.select_dtypes(include=['float64']).columns
ds_jobs_transformed[float_columns] = ds_jobs_transformed[float_columns].astype('float16')

# Convert object columns to category or bool
for col in ds_jobs_transformed.select_dtypes(include=['object']).columns:
    unique_values = ds_jobs_transformed[col].nunique()
    if unique_values == 2:
        ds_jobs_transformed[col] = ds_jobs_transformed[col].astype('bool')
    else:
        ds_jobs_transformed[col] = ds_jobs_transformed[col].astype('category')

# Convert 'job_change' column to bool
ds_jobs_transformed['job_change'] = ds_jobs_transformed['job_change'].map({0.0: False, 1.0: True}).astype('bool')

# Converting ordered categories

In [49]:
from pandas.api.types import CategoricalDtype

# Define the ordered list for work experience
experience_order = [
    '<1', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '>20'
]

# Convert the 'experience' column to an ordered categorical type
experience_dtype = CategoricalDtype(categories=experience_order, ordered=True)
ds_jobs_transformed['experience'] = ds_jobs_transformed['experience'].astype(experience_dtype)

# Define the ordered list for 'education_level'
education_order = ['Primary School', 'High School', 'Graduate', 'Masters', 'Phd']

# Convert the 'education_level' column to an ordered categorical type
education_dtype = CategoricalDtype(categories=education_order, ordered=True)
ds_jobs_transformed['education_level'] = ds_jobs_transformed['education_level'].astype(education_dtype)

# Define the ordered list for 'last_new_job'
last_new_job_order = ['never', '1', '2', '3', '4', '>4']

# Convert the 'last_new_job' column to an ordered categorical type
last_new_job_dtype = CategoricalDtype(categories=last_new_job_order, ordered=True)
ds_jobs_transformed['last_new_job'] = ds_jobs_transformed['last_new_job'].astype(last_new_job_dtype)

# Define the ordered list for 'enrolled_university'
enrolled_university_order = ['no_enrollment', 'Part time course', 'Full time course']

# Convert the 'enrolled_university' column to an ordered categorical type
enrolled_university_dtype = CategoricalDtype(categories=enrolled_university_order, ordered=True)
ds_jobs_transformed['enrolled_university'] = ds_jobs_transformed['enrolled_university'].astype(enrolled_university_dtype)

In [50]:
import pandas as pd

# Define the ordered categories for 'company_size'
company_size_order = {
    '1-49': 1,
    '50-99': 2,
    '100-499': 3,
    '500-999': 4,
    '1000-4999': 5,
    '5000-9999': 6,
    '10000+': 7
}

# Create a CategoricalDtype with the specified order
company_size_dtype = pd.CategoricalDtype(categories=company_size_order.keys(), ordered=True)

# Convert the 'company_size' column to the ordered categorical type
ds_jobs_transformed['company_size'] = ds_jobs_transformed['company_size'].astype(company_size_dtype)

# Filtering on ordered categorical columns

In [51]:
# Filter the DataFrame to only contain students with 10 or more years of experience at companies with at least 1000 employees
ds_jobs_transformed = ds_jobs_transformed[
    (ds_jobs_transformed['experience'] >= '10') & 
    (ds_jobs_transformed['company_size'].isin(['1000-4999', '5000-9999', '10000+']))
]